In [2]:
import numpy as np
import pandas as pd
from aequilibrae.paths.public_transport import HyperpathGenerating
from numba import jit

RS = 124  # random seed

In [3]:
def create_vertices(n):
    x = np.linspace(0, 1, n)
    y = np.linspace(0, 1, n)
    xv, yv = np.meshgrid(x, y, indexing="xy")
    vertices = pd.DataFrame()
    vertices["x"] = xv.ravel()
    vertices["y"] = yv.ravel()
    return vertices

n = 3
vertices = create_vertices(n)
vertices

,x,y
0,0.0,0.0
1,0.5,0.0
2,1.0,0.0
3,0.0,0.5
4,0.5,0.5
5,1.0,0.5
6,0.0,1.0
7,0.5,1.0
8,1.0,1.0


In [4]:
@jit
def create_edges_numba(n):
    m = 2 * n * (n - 1)
    tail = np.zeros(m, dtype=np.uint32)
    head = np.zeros(m, dtype=np.uint32)
    k = 0
    for i in range(n - 1):
        for j in range(n):
            tail[k] = i + j * n
            head[k] = i + 1 + j * n
            k += 1
            tail[k] = j + i * n
            head[k] = j + (i + 1) * n
            k += 1
    return tail, head

def create_edges(n, seed=124):
    tail, head = create_edges_numba(n)
    edges = pd.DataFrame()
    edges["tail"] = tail
    edges["head"] = head
    m = len(edges)
    rng = np.random.default_rng(seed=seed)
    edges["trav_time"] = rng.uniform(0.0, 1.0, m)
    edges["delay_base"] = rng.uniform(0.0, 1.0, m)
    edges['var_1_c'] = rng.uniform(0.0, 1.0, m)
    return edges

In [5]:
edges = create_edges(n, seed=RS)

In [6]:
alpha = 10.0

delay_base = edges.delay_base.values
indices = np.where(delay_base == 0.0)
delay_base[indices] = 1.0 # use this to prevent an error?
freq_base = 1.0 / delay_base
freq_base[indices] = np.inf
edges["freq_base"] = freq_base

if alpha == 0.0:
    edges["freq"] = np.inf
else:
    edges["freq"] = edges.freq_base / alpha

In [7]:
edges['skim_1'] = 1

In [10]:
edges['skim_1'] = [1,2]*int(edges.shape[0]/2)

In [11]:
edges

,tail,head,trav_time,delay_base,var_1_c,freq_base,freq,skim_1
0,0,1,0.785253,0.989463,0.300653,1.010649,0.101065,1
1,0,3,0.785859,0.257111,0.366672,3.889374,0.388937,2
2,3,4,0.969136,0.715765,0.148790,1.397107,0.139711,1
3,1,4,0.748060,0.505885,0.348341,1.976733,0.197673,2
4,6,7,0.655551,0.664111,0.597130,1.505772,0.150577,1
5,2,5,0.938885,0.702342,0.994164,1.423807,0.142381,2
6,1,2,0.178614,0.052080,0.185251,19.201144,1.920114,1
7,3,6,0.588647,0.060096,0.993134,16.639908,1.663991,2
8,4,5,0.442799,0.945353,0.785619,1.057806,0.105781,1
9,4,7,0.348847,0.250350,0.448143,3.994401,0.399440,2


In [12]:
# Spiess & Florian
sf = HyperpathGenerating(
    edges, tail="tail", head="head", trav_time="trav_time", freq="freq", skim_cols = ['skim_1']
)


In [13]:
dest = n * n - 1
sf.run(origin=0, destination=dest, volume=1.0)

[12, 3, 0, 0, 0, 0, 0, 0]
tail_vert_idx 5
edge_idx 11
beta_skim 0.0
skim_i 0.0
skim_j 2.0
skim_i_new 2.0
f_a 0.3623668661968535
f_i 0.0
f_a + f_i 0.3623668661968535

tail_vert_idx 7
edge_idx 10
beta_skim 0.0
skim_i 0.0
skim_j 1.0
skim_i_new 1.0
f_a 0.2464304869512155
f_i 0.0
f_a + f_i 0.2464304869512155

tail_vert_idx 4
edge_idx 8
beta_skim 0.0
skim_i 0.0
skim_j 3.0
skim_i_new 3.0
f_a 0.10578062255888714
f_i 0.0
f_a + f_i 0.10578062255888714

tail_vert_idx 2
edge_idx 5
beta_skim 0.0
skim_i 0.0
skim_j 4.0
skim_i_new 4.0
f_a 0.1423806817244761
f_i 0.0
f_a + f_i 0.1423806817244761

tail_vert_idx 4
edge_idx 9
beta_skim 0.3173418676766614
skim_i 3.0
skim_j 3.0
skim_i_new 3.0
f_a 0.3994401011051945
f_i 0.10578062255888714
f_a + f_i 0.5052207236640817

tail_vert_idx 6
edge_idx 4
beta_skim 0.0
skim_i 0.0
skim_j 2.0
skim_i_new 2.0
f_a 0.1505771644152682
f_i 0.0
f_a + f_i 0.1505771644152682

tail_vert_idx 1
edge_idx 3
beta_skim 0.0
skim_i 0.0
skim_j 5.0
skim_i_new 5.0
f_a 0.19767333768469803
f_i

In [14]:
edges_df = sf._edges

for i in range(sf._indptr[dest], sf._indptr[dest+1]):
    print(i)
    edge_idx = sf._edge_idx[i]
    print(edge_idx)
    print(sf._edges.trav_time[edge_idx])
    display(edges_df[edges_df.edge_idx == edge_idx])
edges_df[edges_df['head'] == 99]

10
10
0.3309294955246901


,tail,head,trav_time,freq,skim_1,edge_idx,volume
10,7,8,0.330929,0.24643,1,10,0.796094


11
11
0.15936868286018258


,tail,head,trav_time,freq,skim_1,edge_idx,volume
11,5,8,0.159369,0.362367,2,11,0.203906


,tail,head,trav_time,freq,skim_1,edge_idx,volume


In [15]:
sf._edges

,tail,head,trav_time,freq,skim_1,edge_idx,volume
0,0,1,0.785253,0.101065,1,0,0.206254
1,0,3,0.785859,0.388937,2,1,0.793746
2,3,4,0.969136,0.139711,1,2,0.061482
3,1,4,0.748060,0.197673,2,3,0.019252
4,6,7,0.655551,0.150577,1,4,0.732264
5,2,5,0.938885,0.142381,2,5,0.187002
6,1,2,0.178614,1.920114,1,6,0.187002
7,3,6,0.588647,1.663991,2,7,0.732264
8,4,5,0.442799,0.105781,1,8,0.016904
9,4,7,0.348847,0.399440,2,9,0.063830


In [16]:
edges_df = sf._edges
edges_df.loc[edges_df['head'] == 8,]

,tail,head,trav_time,freq,skim_1,edge_idx,volume
10,7,8,0.330929,0.246430,1,10,0.796094
11,5,8,0.159369,0.362367,2,11,0.203906


In [17]:
sf.u_i_vec

array([15.01319095, 11.16968936, 10.88131216, 12.45090564,  6.4289669 ,
        2.91900289, 11.68553301,  4.38886897,  0.        ])

In [18]:
sf.skim_i_vec

array([6., 5., 4., 4., 3., 2., 2., 1., 0.])